# A–E Repeated-Measures Analysis with Sphericity + Greenhouse–Geisser

This notebook upgrades the current image-level analysis.

For every AOI × metric it:
- keeps only images complete across A–E
- runs repeated-measures ANOVA with `Img` as the repeated unit
- tests sphericity with Mauchly's test
- computes Greenhouse–Geisser epsilon
- reports uncorrected and GG-corrected p-values
- applies BH-FDR across the 24 omnibus tests
- runs paired post-hoc tests for significant omnibus effects
- computes paired effect size `d_z`
- computes bootstrap 95% confidence intervals
- applies BH-FDR across post-hoc tests

Input:
`data/A_to_E_long_image_level_data.csv`


In [1]:
# Install once if needed:
# %pip install pandas numpy scipy statsmodels pingouin

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
import pingouin as pg


In [2]:
LONG_CSV = "data/A_to_E_long_image_level_data.csv"

AOIS = ["CTA", "Headline", "Product"]

METRICS = [
    "VAI (%)",
    "TTFF (s)",
    "Time spent (s)",
    "Fixations",
    "Fix. duration (s)",
    "FFD (s)",
    "K-coefficient",
    "Revisits",
]

CONDITIONS = ["A", "B", "C", "D", "E"]

ALPHA = 0.05
N_BOOTSTRAP = 5000
RANDOM_SEED = 42

OMNIBUS_OUT = "A_to_E_GG_omnibus.csv"
POSTHOC_OUT = "A_to_E_GG_posthoc.csv"
POSTHOC_SIG_OUT = "A_to_E_GG_posthoc_significant.csv"
MASTER_OUT = "A_to_E_GG_master_summary.csv"


In [3]:
if not Path(LONG_CSV).exists():
    raise FileNotFoundError(f"Could not find {LONG_CSV}")

df = pd.read_csv(LONG_CSV)
df.columns = [str(c).replace("**", "").strip() for c in df.columns]
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["value"])

print("Rows:", len(df))
print("AOIs:", sorted(df["AOI"].dropna().unique()))
print("Metrics:", sorted(df["metric"].dropna().unique()))


Rows: 2320
AOIs: ['CTA', 'Headline', 'Product']
Metrics: ['FFD (s)', 'Fix. duration (s)', 'Fixations', 'K-coefficient', 'Revisits', 'TTFF (s)', 'Time spent (s)', 'VAI (%)']


In [4]:
def complete_wide(aoi, metric):
    sub = df[
        (df["AOI"] == aoi)
        & (df["metric"] == metric)
        & (df["condition"].isin(CONDITIONS))
    ].copy()

    return (
        sub.pivot(index="Img", columns="condition", values="value")
        .reindex(columns=CONDITIONS)
        .dropna()
    )


def wide_to_long(wide):
    return (
        wide.reset_index()
        .melt(
            id_vars="Img",
            value_vars=CONDITIONS,
            var_name="condition",
            value_name="value",
        )
    )


def paired_dz(x, y):
    d = np.asarray(y, float) - np.asarray(x, float)
    d = d[np.isfinite(d)]

    if len(d) < 2:
        return np.nan

    sd = np.std(d, ddof=1)

    if sd == 0:
        return 0.0

    return np.mean(d) / sd


def bootstrap_paired(x, y, seed):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if n < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)

    obs_diff = np.mean(y - x)
    obs_dz = paired_dz(x, y)

    diff_boot = []
    dz_boot = []

    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        xb, yb = x[idx], y[idx]

        diff_boot.append(np.mean(yb - xb))

        dzb = paired_dz(xb, yb)
        if np.isfinite(dzb):
            dz_boot.append(dzb)

    diff_lo, diff_hi = np.percentile(diff_boot, [2.5, 97.5])
    dz_lo, dz_hi = np.percentile(dz_boot, [2.5, 97.5])

    return obs_diff, diff_lo, diff_hi, obs_dz, dz_lo, dz_hi


## 1. Omnibus repeated-measures ANOVA

In [5]:
omnibus_rows = []

for aoi in AOIS:
    for metric in METRICS:
        wide = complete_wide(aoi, metric)
        n = len(wide)

        if n < 3:
            continue

        long = wide_to_long(wide)

        fit = AnovaRM(
            data=long,
            depvar="value",
            subject="Img",
            within=["condition"],
        ).fit()

        tab = fit.anova_table

        F = float(tab.loc["condition", "F Value"])
        df1 = float(tab.loc["condition", "Num DF"])
        df2 = float(tab.loc["condition", "Den DF"])
        p_unc = float(tab.loc["condition", "Pr > F"])

        try:
            sph = pg.sphericity(wide[CONDITIONS], method="mauchly")
            mauchly_W = float(sph.W)
            mauchly_p = float(sph.pval)
            sphericity_met = bool(sph.spher)
        except Exception:
            mauchly_W = np.nan
            mauchly_p = np.nan
            sphericity_met = np.nan

        try:
            eps_gg = float(pg.epsilon(wide[CONDITIONS], correction="gg"))
        except Exception:
            eps_gg = np.nan

        if np.isfinite(eps_gg):
            df1_gg = eps_gg * df1
            df2_gg = eps_gg * df2
            p_gg = stats.f.sf(F, df1_gg, df2_gg)
        else:
            df1_gg = np.nan
            df2_gg = np.nan
            p_gg = np.nan

        eta_p2 = (F * df1) / ((F * df1) + df2)

        omnibus_rows.append({
            "AOI": aoi,
            "metric": metric,
            "n_complete_images": n,
            "mean_A": wide["A"].mean(),
            "mean_B": wide["B"].mean(),
            "mean_C": wide["C"].mean(),
            "mean_D": wide["D"].mean(),
            "mean_E": wide["E"].mean(),
            "F": F,
            "df1": df1,
            "df2": df2,
            "p_uncorrected": p_unc,
            "mauchly_W": mauchly_W,
            "mauchly_p": mauchly_p,
            "sphericity_met": sphericity_met,
            "epsilon_GG": eps_gg,
            "df1_GG": df1_gg,
            "df2_GG": df2_gg,
            "p_GG": p_gg,
            "partial_eta_sq": eta_p2,
        })

omnibus = pd.DataFrame(omnibus_rows)

valid = omnibus["p_GG"].notna()
omnibus["q_GG_FDR"] = np.nan
omnibus["significant_GG_FDR"] = False

if valid.any():
    reject, qvals, _, _ = multipletests(
        omnibus.loc[valid, "p_GG"],
        alpha=ALPHA,
        method="fdr_bh",
    )

    omnibus.loc[valid, "q_GG_FDR"] = qvals
    omnibus.loc[valid, "significant_GG_FDR"] = reject

omnibus.to_csv(OMNIBUS_OUT, index=False)
omnibus.sort_values("q_GG_FDR")


,AOI,metric,n_complete_images,mean_A,mean_B,mean_C,mean_D,mean_E,F,df1,...,mauchly_W,mauchly_p,sphericity_met,epsilon_GG,df1_GG,df2_GG,p_GG,partial_eta_sq,q_GG_FDR,significant_GG_FDR
20,Product,Fix. duration (s),17,0.260000,0.206471,0.215882,0.204118,0.215882,13.008597,4.0,...,0.234137,0.013517,False,0.675060,2.700240,43.203834,0.000007,0.448439,0.000161,True
12,Headline,Fix. duration (s),17,0.272353,0.216471,0.214118,0.223529,0.221765,6.896861,4.0,...,0.242119,0.015970,False,0.593820,2.375281,38.004489,0.001702,0.301214,0.011151,True
21,Product,FFD (s),17,0.257647,0.208235,0.218235,0.200588,0.216471,7.276864,4.0,...,0.201480,0.006296,False,0.539660,2.158639,34.538227,0.001858,0.312622,0.011151,True
8,Headline,VAI (%),17,38.558824,48.094118,51.623529,41.476471,40.935294,5.733294,4.0,...,0.473037,0.293295,True,0.810858,3.243434,51.894938,0.001414,0.263802,0.011151,True
13,Headline,FFD (s),17,0.277647,0.220588,0.221176,0.225882,0.224706,5.455960,4.0,...,0.397350,0.151749,True,0.761544,3.046176,48.738811,0.002468,0.254286,0.011847,True
15,Headline,Revisits,17,0.545882,0.794118,1.030588,0.652941,0.607647,5.206821,4.0,...,0.266726,0.025630,False,0.652335,2.609340,41.749445,0.005374,0.245526,0.021496,True
3,CTA,Fixations,17,17.529412,23.294118,34.352941,22.411765,28.470588,4.022171,4.0,...,0.508029,0.373103,True,0.802913,3.211651,51.386418,0.010520,0.200886,0.036068,True
0,CTA,VAI (%),17,21.635294,26.682353,34.482353,26.129412,28.388235,3.283599,4.0,...,0.512225,0.383118,True,0.777244,3.108975,49.743599,0.026923,0.170279,0.064615,False
2,CTA,Time spent (s),17,0.497647,0.521765,0.637647,0.439412,0.639412,3.623663,4.0,...,0.277531,0.030998,False,0.657762,2.631049,42.096787,0.024895,0.184658,0.064615,False
11,Headline,Fixations,17,36.882353,54.764706,64.294118,47.411765,44.705882,4.308237,4.0,...,0.161119,0.001934,False,0.482158,1.928631,30.858089,0.023500,0.212142,0.064615,False


## 2. Paired post-hoc tests + effect sizes

In [6]:
posthoc_rows = []
seed_counter = 0

sig_omnibus = omnibus[
    omnibus["significant_GG_FDR"] == True
].copy()

for _, row in sig_omnibus.iterrows():
    aoi = row["AOI"]
    metric = row["metric"]

    wide = complete_wide(aoi, metric)

    for g1, g2 in combinations(CONDITIONS, 2):
        x = wide[g1].values.astype(float)
        y = wide[g2].values.astype(float)

        t, p = stats.ttest_rel(x, y)

        (
            mean_diff,
            diff_lo,
            diff_hi,
            dz,
            dz_lo,
            dz_hi,
        ) = bootstrap_paired(
            x,
            y,
            RANDOM_SEED + seed_counter,
        )

        seed_counter += 1

        posthoc_rows.append({
            "AOI": aoi,
            "metric": metric,
            "group1": g1,
            "group2": g2,
            "n": len(wide),
            "mean_group1": np.mean(x),
            "mean_group2": np.mean(y),
            "t": t,
            "p_raw": p,
            "mean_diff_group2_minus_group1": mean_diff,
            "mean_diff_ci_low": diff_lo,
            "mean_diff_ci_high": diff_hi,
            "dz": dz,
            "dz_ci_low": dz_lo,
            "dz_ci_high": dz_hi,
        })

posthoc = pd.DataFrame(posthoc_rows)

if len(posthoc):
    reject, qvals, _, _ = multipletests(
        posthoc["p_raw"],
        alpha=ALPHA,
        method="fdr_bh",
    )

    posthoc["q_posthoc_global_FDR"] = qvals
    posthoc["significant_posthoc_global_FDR"] = reject

posthoc.to_csv(POSTHOC_OUT, index=False)

posthoc_sig = posthoc[
    posthoc["significant_posthoc_global_FDR"] == True
].copy()

posthoc_sig.to_csv(POSTHOC_SIG_OUT, index=False)

posthoc_sig


,AOI,metric,group1,group2,n,mean_group1,mean_group2,t,p_raw,mean_diff_group2_minus_group1,mean_diff_ci_low,mean_diff_ci_high,dz,dz_ci_low,dz_ci_high,q_posthoc_global_FDR,significant_posthoc_global_FDR
1,CTA,Fixations,A,C,17,17.529412,34.352941,-3.559392,0.002614,16.823529,7.764706,25.588235,0.863279,0.414286,1.610224,0.015249,True
7,CTA,Fixations,C,D,17,34.352941,22.411765,2.854805,0.011466,-11.941176,-20.117647,-4.000000,-0.692392,-1.420546,-0.241335,0.034898,True
11,Headline,VAI (%),A,C,17,38.558824,51.623529,-4.150804,0.000752,13.064706,7.276176,19.329559,1.006718,0.587836,1.889026,0.005862,True
17,Headline,VAI (%),C,D,17,51.623529,41.476471,4.104391,0.000829,-10.147059,-14.712500,-5.270441,-0.995461,-1.819731,-0.534418,0.005862,True
18,Headline,VAI (%),C,E,17,51.623529,40.935294,4.314767,0.000534,-10.688235,-15.376618,-5.923235,-1.046485,-1.759898,-0.617306,0.005862,True
20,Headline,Fix. duration (s),A,B,17,0.272353,0.216471,3.187318,0.005728,-0.055882,-0.088235,-0.022353,-0.773038,-1.603859,-0.307996,0.024432,True
21,Headline,Fix. duration (s),A,C,17,0.272353,0.214118,3.424755,0.003474,-0.058235,-0.090000,-0.024706,-0.830625,-1.760914,-0.341702,0.017369,True
22,Headline,Fix. duration (s),A,D,17,0.272353,0.223529,2.913626,0.010150,-0.048824,-0.080588,-0.017647,-0.706658,-1.586882,-0.245017,0.032295,True
23,Headline,Fix. duration (s),A,E,17,0.272353,0.221765,3.300681,0.004513,-0.050588,-0.080000,-0.021765,-0.800533,-1.614130,-0.362347,0.021060,True
30,Headline,FFD (s),A,B,17,0.277647,0.220588,3.451105,0.003286,-0.057059,-0.086471,-0.024118,-0.837016,-2.045958,-0.316491,0.017369,True


## 3. Compact master summary

In [7]:
def summarize_posthoc(aoi, metric):
    sub = posthoc_sig[
        (posthoc_sig["AOI"] == aoi)
        & (posthoc_sig["metric"] == metric)
    ]

    if len(sub) == 0:
        return ""

    out = []

    for _, r in sub.iterrows():
        out.append(
            f'{r["group1"]}-{r["group2"]}: '
            f'q={r["q_posthoc_global_FDR"]:.4f}, '
            f'dz={r["dz"]:.2f}'
        )

    return "; ".join(out)


master = omnibus.copy()

master["significant_posthoc"] = master.apply(
    lambda r: summarize_posthoc(r["AOI"], r["metric"]),
    axis=1,
)

master.to_csv(MASTER_OUT, index=False)
master


,AOI,metric,n_complete_images,mean_A,mean_B,mean_C,mean_D,mean_E,F,df1,...,mauchly_p,sphericity_met,epsilon_GG,df1_GG,df2_GG,p_GG,partial_eta_sq,q_GG_FDR,significant_GG_FDR,significant_posthoc
0,CTA,VAI (%),17,21.635294,26.682353,34.482353,26.129412,28.388235,3.283599,4.0,...,0.383118,True,0.777244,3.108975,49.743599,0.026923,0.170279,0.064615,False,
1,CTA,TTFF (s),16,3.123750,2.768125,2.516875,2.923750,3.076875,3.237228,4.0,...,0.066206,True,0.623641,2.494566,37.418485,0.040602,0.177507,0.070811,False,
2,CTA,Time spent (s),17,0.497647,0.521765,0.637647,0.439412,0.639412,3.623663,4.0,...,0.030998,False,0.657762,2.631049,42.096787,0.024895,0.184658,0.064615,False,
3,CTA,Fixations,17,17.529412,23.294118,34.352941,22.411765,28.470588,4.022171,4.0,...,0.373103,True,0.802913,3.211651,51.386418,0.010520,0.200886,0.036068,True,"A-C: q=0.0152, dz=0.86; C-D: q=0.0349, dz=-0.69"
4,CTA,Fix. duration (s),16,0.274375,0.231875,0.232500,0.218750,0.240000,3.182437,4.0,...,0.047051,False,0.640728,2.562911,38.443662,0.041306,0.175028,0.070811,False,
5,CTA,FFD (s),16,0.276875,0.232500,0.240000,0.220000,0.251250,2.784858,4.0,...,0.038905,False,0.649715,2.598859,38.982888,0.060747,0.156586,0.091121,False,
6,CTA,K-coefficient,16,0.194375,0.234375,0.305000,-0.037500,0.270625,0.972355,4.0,...,0.166620,True,0.713488,2.853953,42.809289,0.411331,0.060877,0.429215,False,
7,CTA,Revisits,17,0.248235,0.352353,0.548235,0.344706,0.420588,3.145654,4.0,...,0.326257,True,0.739245,2.956981,47.311698,0.034269,0.164301,0.070811,False,
8,Headline,VAI (%),17,38.558824,48.094118,51.623529,41.476471,40.935294,5.733294,4.0,...,0.293295,True,0.810858,3.243434,51.894938,0.001414,0.263802,0.011151,True,"A-C: q=0.0059, dz=1.01; C-D: q=0.0059, dz=-1.0..."
9,Headline,TTFF (s),17,1.598824,1.767647,1.576471,1.900588,1.907059,1.481704,4.0,...,0.481376,True,0.744776,2.979103,47.665655,0.231611,0.084757,0.304788,False,


## Main interpretation columns

- `mauchly_p < .05` → evidence that sphericity is violated.
- `epsilon_GG` → Greenhouse–Geisser correction factor.
- `p_GG` → Greenhouse–Geisser-corrected omnibus p-value.
- `q_GG_FDR` → BH-FDR-adjusted omnibus p-value across all 24 AOI × metric tests.
- `partial_eta_sq` → omnibus effect size.
- `dz` → paired standardized effect size for each pairwise comparison.
- `mean_diff_group2_minus_group1` → raw paired difference in the original units.
